# Validating a General-Relativistic Ray Tracer Against Event Horizon Telescope Observations

**Ethan Knox** &nbsp;|&nbsp; [github.com/ethank5149](https://github.com/ethank5149) &nbsp;|&nbsp; ethank5149@gmail.com

---

## 1. Introduction

In April 2019, the Event Horizon Telescope (EHT) Collaboration published the first image of a black hole — M87\*, the supermassive object at the center of Messier 87 — revealing a bright asymmetric ring of diameter $42 \pm 3\;\mu\text{as}$ consistent with the predicted shadow of a Kerr black hole. In May 2022, the EHT released a second image: Sgr A\*, the $4 \times 10^6\;M_\odot$ black hole at the center of our Milky Way, with a shadow angular diameter of $48.7 \pm 7\;\mu\text{as}$.

In this notebook I validate **[Nulltracer](https://github.com/ethank5149/nulltracer)** — a GPU-accelerated ray tracer I built — against these published EHT measurements. The CUDA kernels implement the Fuerst & Wu (2004) Hamiltonian formulation of null geodesics in the Kerr-Newman metric, cross-verified against the Odyssey (Pu et al. 2016), GRay (Chan et al. 2013), and RAPTOR (Bronzwaer et al. 2018) GPU ray-tracing codes, and validated against the analytic shadow boundary curve of Bardeen (1973).

**Scope of this notebook:** geometry and shadow validation only — no GRMHD plasma, no polarized radiative transfer, no interstellar scattering. Those limitations are discussed in §12.


In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

import nulltracer as nt

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.figsize': (10, 8), 'font.size': 12, 'font.family': 'serif',
    'axes.titlesize': 14, 'axes.labelsize': 12,
    'savefig.dpi': 150, 'savefig.facecolor': 'black',
})

# ── GPU probe ─────────────────────────────────────────────────
props = cp.cuda.runtime.getDeviceProperties(0)
gpu_name = props['name'].decode() if isinstance(props['name'], bytes) else props['name']
print(f"GPU: {gpu_name} ({props['totalGlobalMem'] / 1e9:.1f} GB)")

# Load background skymap once; falls back gracefully if missing
try:
    nt.load_skymap("assets/starmap_gaia_4k.jpg")
    print("Skymap: starmap_gaia_4k.jpg loaded")
except Exception as exc:  # noqa: BLE001
    print(f"Skymap unavailable ({exc!s}); bg_mode=3 will fall back to procedural stars")

# ── Renderer + kernel pre-compilation ─────────────────────────
# Construct and initialize BEFORE using — the previous version of this cell
# referenced `renderer` before it was defined (NameError).
renderer = nt.CudaRenderer()
renderer.initialize()
nt.compile_all(verbose=False)           # front-load CUDA compilation latency
print(f"Kernels compiled: {', '.join(nt.available_methods())}")


### Hero renders

Two canonical views produced with the free `nt.render_frame()` API, which uses `auto_steps()` to size the integration budget from `obs_dist` and `step_size`. The dict-based `CudaRenderer.render_frame()` API now also accepts the shorthand `incl` alias and auto-sizes steps when omitted.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Gargantua-like: high spin, edge-on
img_g, info_g = nt.render_frame(
    spin=0.95, inclination_deg=80,
    width=1024, height=1024, fov=10.0, obs_dist=40,
    step_size=0.25, method='rkdp8',
    bg_mode=3, disk_temp=1.3,
)
axes[0].imshow(img_g)
axes[0].set_title('Kerr ($a=0.95$, $\\theta=80°$) — Gargantua-like', fontsize=13, pad=10)
axes[0].axis('off')

# M87*-like: high spin, low inclination
img_m, info_m = nt.render_frame(
    spin=0.94, inclination_deg=30,
    width=1024, height=1024, fov=10.0, obs_dist=50,
    step_size=0.25, method='rkdp8',
    bg_mode=3, disk_temp=0.6,
)
axes[1].imshow(img_m)
axes[1].set_title('Kerr ($a=0.94$, $\\theta=30°$) — M87*-like', fontsize=13, pad=10)
axes[1].axis('off')

plt.tight_layout()
plt.show()


---

## 2. Physics: the Kerr Metric and Null Geodesics

### 2.1 The Kerr metric

A rotating black hole of mass $M$ and angular momentum $J = aM$ is described by the Kerr metric in Boyer–Lindquist coordinates $(t, r, \theta, \phi)$. In geometric units $G = c = M = 1$, with $\Sigma = r^2 + a^2\cos^2\theta$ and $\Delta = r^2 - 2r + a^2$:

$$ds^2 = -\frac{\Delta - a^2\sin^2\theta}{\Sigma}\,dt^2 - \frac{4ar\sin^2\theta}{\Sigma}\,dt\,d\phi + \frac{\Sigma}{\Delta}\,dr^2 + \Sigma\,d\theta^2 + \frac{A\sin^2\theta}{\Sigma}\,d\phi^2$$

where $A = (r^2 + a^2)^2 - a^2\Delta\sin^2\theta$. The event horizon is at $r_+ = 1 + \sqrt{1 - a^2}$.

### 2.2 Carter separation and constants of motion

Carter (1968) showed that the Hamilton-Jacobi equation separates in the Kerr metric, yielding three constants of motion for null geodesics: energy $E = -p_t$, axial angular momentum $L_z = p_\phi$, and the Carter constant $\mathcal{Q} = p_\theta^2 + \cos^2\theta(a^2 E^2 - L_z^2/\sin^2\theta)$. Setting $E = 1$ (absorbing the energy into the affine parameter), define dimensionless impact parameters $\xi = L_z$ and $\eta = \mathcal{Q}$.

The separated radial and angular potentials are

$$R(r) = r^4 + (a^2 - \xi^2 - \eta)\,r^2 + 2\bigl[(\xi - a)^2 + \eta\bigr]\,r - a^2\eta,$$

$$\Theta(\theta) = \eta + a^2\cos^2\theta - \xi^2\cot^2\theta.$$

### 2.3 Hamiltonian equations of motion

Following Fuerst & Wu (2004) and the Odyssey GPU code (Pu et al. 2016), the state vector is $(r, \theta, \phi, p_r, p_\theta)$ — five ODEs, since $p_t = -1$ and $p_\phi = \xi$ are constants. With $\kappa = \eta + \xi^2 + a^2$,

| Equation | Explicit form |
|---|---|
| $\dot{r}$ | $\Delta\, p_r / \Sigma$ |
| $\dot{\theta}$ | $p_\theta / \Sigma$ |
| $\dot{\phi}$ | $\bigl[2ar + (\Sigma - 2r)\,\xi / \sin^2\theta\bigr] / (\Sigma\Delta)$ |
| $\dot{p}_r$ | $\bigl[-\kappa(r-1) + 2r(r^2+a^2) - 2a\xi\bigr] / (\Sigma\Delta) - 2p_r^2(r-1)/\Sigma$ |
| $\dot{p}_\theta$ | $\sin\theta\cos\theta\bigl[\xi^2/\sin^4\theta - a^2\bigr] / \Sigma$ |

At turning points ($R(r) = 0$ or $\Theta(\theta) = 0$), the momenta $p_r$ and $p_\theta$ pass smoothly through zero — no sign-tracking is needed, eliminating branch divergence on the GPU.

### 2.4 Impact parameters (Bardeen 1973)

For a pixel at screen coordinates $(\alpha, \beta)$ in units of $M$, the conserved quantities are

$$\xi = -\alpha\sin\theta_{\text{obs}}, \qquad \eta = \beta^2 + (\alpha^2 - a^2)\cos^2\theta_{\text{obs}}.$$

The initial radial momentum is $p_r = -\sqrt{R(r_{\text{obs}})} / \Delta_{\text{obs}}$ (negative for backward/inward tracing), and $p_\theta = \beta$ (sign matches the polar direction of the arriving photon).


## 3. Implementation

### 3.1 ISCO (Bardeen, Press & Teukolsky 1972)

Innermost stable circular orbit for a prograde equatorial test particle. The Schwarzschild limit is $r_{\rm ISCO}(a{=}0) = 6M$; the extremal Kerr limit is $r_{\rm ISCO}(a{\to}1) = M$.

In [ ]:
for a_val, expected in [(0.0, 6.0), (0.5, 4.233), (0.9, 2.321), (0.998, 1.237)]:
    got = nt.isco_kerr(a_val)
    print(f"ISCO(a={a_val:5.3f}): {got:7.4f} M  (expected {expected:7.4f} M, "
          f"Δ = {abs(got - expected):.2e})")


### 3.2 Analytic shadow boundary (Bardeen 1973, Chandrasekhar 1983)

The shadow edge corresponds to photons on unstable spherical orbits satisfying $R(r) = 0$ and $dR/dr = 0$. The critical impact parameters as functions of the orbit radius $r$ are

$$\xi_c(r) = -\frac{r^3 - 3r^2 + a^2 r + a^2}{a(r - 1)}, \qquad \eta_c(r) = -\frac{r^3(r^3 - 6r^2 + 9r - 4a^2)}{a^2(r-1)^2}.$$

Photon orbit radii range from $r_{\text{ph}}^- = 2[1 + \cos(\tfrac{2}{3}\arccos(-a))]$ (prograde) to $r_{\text{ph}}^+ = 2[1 + \cos(\tfrac{2}{3}\arccos(+a))]$ (retrograde).

In [ ]:
# Schwarzschild check: shadow is a circle of radius 3√3 ≈ 5.196 M.
alpha_sch, beta_p, beta_m = nt.shadow_boundary(1e-6, np.pi / 2)
r_shadow_sch = max(
    np.sqrt(alpha_sch ** 2 + beta_p ** 2).max(),
    np.sqrt(alpha_sch ** 2 + beta_m ** 2).max(),
)
expected = 3.0 * np.sqrt(3.0)
print(f"Analytic shadow radius (a=1e-6, θ=π/2): {r_shadow_sch:.6f} M")
print(f"Schwarzschild limit 3√3:                {expected:.6f} M")
print(f"Relative error: {abs(r_shadow_sch - expected) / expected:.2e}")
assert abs(r_shadow_sch - expected) / expected < 1e-4


### 3.3 CUDA ray-tracing kernels

The kernels implement the Fuerst & Wu (2004) Hamiltonian ODEs from §2.3 with seven integrators available (RK4, DP8, and Kahan–Li / Tao–Yoshida symplectic variants). Each GPU thread traces one null geodesic backward from the camera. All geodesic integration uses `float64`; colour arithmetic is done in `float32`; the final image is `uint8` sRGB.

**Equation sources (line-by-line):**
- Impact parameters: Bardeen (1973), §4.
- Initial $p_r$: §5 of the same reference, with $R(r)$ from §2.
- RHS ODEs: Fuerst & Wu (2004); Odyssey (Pu et al. 2016, Eq. 14).
- Disk redshift (r ≥ ISCO): $g = \sqrt{1 - 3/r + 2a/r^{3/2}} / (1 - \Omega\xi_\gamma)$; inside ISCO the disk kernel switches to the plunging-region form.

In [ ]:
kernels = nt.compile_all(verbose=False)
print('Available integrators:', nt.available_methods())


---

## 4. Validation: numerical shadow vs. analytic boundary

The most rigorous test: overlay the numerically ray-traced shadow boundary against the exact Bardeen (1973) analytic curve. Agreement to sub-pixel precision validates the entire geodesic integration pipeline.

Pixel mapping note: the CUDA kernel uses `ux = (2·(ix+0.5)/W − 1)` to sample pixel centres; the inverse is `px = (ux + 1)·W/2 − 0.5`. With `α = ux · fov · aspect` and `β = uy · fov` (fov expressed as the screen half-width in units of $M$), this gives the pixel mapping used in the overlay below.

In [ ]:
test_cases = [
    (0.0,   90, 'Schwarzschild ($a=0$, $\\theta=90°$)'),
    (0.6,   17, 'Kerr ($a=0.6$, $\\theta=17°$) — M87*-like'),
    (0.998, 50, 'Near-extreme Kerr ($a=0.998$, $\\theta=50°$) — Sgr A*-like'),
    (0.9,   85, 'Kerr ($a=0.9$, $\\theta=85°$) — edge-on'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 14))

for ax_flat, (a_val, theta_deg, label) in zip(axes.flat, test_cases):
    W, H, FOV = 512, 512, 7.0
    params = {
        'spin': a_val, 'inclination': theta_deg,
        'width': W, 'height': H, 'fov': FOV,
        'show_disk': False, 'bg_mode': 1,
        'obs_dist': 500, 'step_size': 0.15, 'method': 'rkdp8',
    }
    timed = renderer.render_frame_timed(params)
    img = np.frombuffer(timed['raw_rgb'], dtype=np.uint8).reshape((H, W, 3))
    kernel_ms = timed['kernel_ms']

    # Analytic boundary (Bardeen 1973)
    a_use = max(a_val, 1e-6)
    alpha_a, beta_p, beta_m = nt.shadow_boundary(a_use, np.radians(theta_deg))

    # Kernel pixel convention: ux = (2·(ix+0.5)/W − 1)  ⇒  ix = (ux+1)·W/2 − 0.5
    # α = ux · FOV · aspect ;  β = uy · FOV  (FOV = screen half-width in M)
    aspect = W / H
    px_alpha = (alpha_a / (FOV * aspect) + 1.0) * 0.5 * W - 0.5
    py_beta_p = (1.0 - beta_p / FOV) * 0.5 * H - 0.5
    py_beta_m = (1.0 - beta_m / FOV) * 0.5 * H - 0.5

    ax_flat.imshow(img)
    ax_flat.plot(px_alpha, py_beta_p, '-', color='cyan', lw=1.5, alpha=0.9,
                 label='Bardeen (1973) analytic')
    ax_flat.plot(px_alpha, py_beta_m, '-', color='cyan', lw=1.5, alpha=0.9)
    ax_flat.set_title(f"{label}\n({kernel_ms:.0f} ms, rkdp8)", fontsize=11)
    ax_flat.legend(fontsize=9, loc='lower right')
    ax_flat.axis('off')

fig.suptitle('Numerical Shadow vs. Analytic Bardeen (1973) Boundary', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()


---

## 5. Qualitative survey: spin × observer inclination

A visual sweep over $(a, \theta_{\rm obs})$ with the disk active. The approaching side brightens via $g^4$ Doppler boosting; frame-dragging warps the disk azimuthally for high $a$.

In [ ]:
spins = [0.0, 0.3, 0.6, 0.9, 0.998]
inclinations = [17, 50, 90]

fig, axes = plt.subplots(len(inclinations), len(spins), figsize=(18, 11))
for i, theta in enumerate(inclinations):
    for j, a in enumerate(spins):
        params = {
            'spin': a, 'inclination': theta,
            'width': 384, 'height': 384,
            'show_disk': True, 'method': 'rk4',
            'bg_mode': 3, 'star_layers': 3, 'disk_temp': 1.2,
        }
        img = np.frombuffer(renderer.render_frame(params),
                            dtype=np.uint8).reshape((384, 384, 3))
        axes[i, j].imshow(img)
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(f'$a = {a}$', fontsize=12)
        if j == 0:
            axes[i, j].set_ylabel(f'$\\theta = {theta}°$', fontsize=12,
                                  rotation=0, labelpad=45, va='center')

fig.suptitle('Kerr Black Hole: Spin × Observer Inclination '
             '(GPU ray-traced, Hamiltonian RK4)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


---

## 6. Shadow observable extraction

`extract_shadow_metrics` thresholds the image, extracts the shadow contour, and fits a circle and an ellipse. The circle fit gives a mean diameter (in units of $M$); the ellipse fit gives circularity $\Delta C = 1 - b/a$ — the EHT observable.

### 6.1 Schwarzschild validation

In [ ]:
params = {
    'spin': 0.0, 'inclination': 90,
    'width': 512, 'height': 512, 'fov': 7.0,
    'obs_dist': 500, 'step_size': 0.15,
    'bg_mode': 1, 'show_disk': False,
}
raw_bytes = renderer.render_frame(params)
img_gray = np.frombuffer(raw_bytes, dtype=np.uint8).reshape((512, 512, 3))
obs_sch = nt.extract_shadow_metrics(img_gray, fov_deg=7.0)

expected = 2.0 * 3.0 * np.sqrt(3.0)                  # ≈ 10.3923 M
if 'error' not in obs_sch:
    print(f"Measured shadow diameter: {obs_sch['diameter_M']:.4f} M")
    print(f"Expected (analytic 2·3√3): {expected:.4f} M")
    rel = abs(obs_sch['diameter_M'] - expected) / expected
    print(f"Relative error:           {rel:.3%}")
    print(f"Circularity ΔC:           {obs_sch['circularity']:.4f}")
else:
    print('Shadow extraction failed:', obs_sch)


---

## 7. Comparison to M87\*

| Observable | Value | Reference |
|---|---|---|
| Mass | $(6.5 \pm 0.7) \times 10^9\;M_\odot$ | EHT I (2019) |
| Distance | $16.8$ Mpc | EHT I (2019) |
| Ring diameter | $42 \pm 3\;\mu\text{as}$ | EHT I (2019) |
| $\Delta C$ | $\lesssim 0.10$ | EHT I (2019) |
| Inclination | $\sim 17°$ | Jet axis |

### 7.1 Unit conversion: $M \to \mu$as

In [ ]:
G = 6.67430e-11; c = 2.99792458e8; M_sun = 1.98892e30
pc = 3.08567758e16; Mpc = pc * 1e6

M87 = dict(mass_kg=6.5e9 * M_sun, dist_m=16.8 * Mpc, incl=17.0,
           ring_uas=42.0, ring_err=3.0)
SgrA = dict(mass_kg=4.0e6 * M_sun, dist_m=8.28e3 * pc, incl=50.0,
            shadow_uas=48.7, shadow_err=7.0, ring_uas=51.8, ring_err=2.3)


def M_to_uas(d_M, mass_kg, dist_m):
    '''Convert a diameter in units of M (gravitational radii) to μas.'''
    return d_M * G * mass_kg / (c ** 2 * dist_m) * (180 / np.pi) * 3600 * 1e6


sch_uas = M_to_uas(2 * 3 * np.sqrt(3), M87['mass_kg'], M87['dist_m'])
print(f"Schwarzschild shadow at M87* scale: {sch_uas:.1f} μas "
      f"(EHT ring: {M87['ring_uas']:.1f} ± {M87['ring_err']:.1f} μas)")


### 7.2 M87* spin sweep

Render at M87's inclination ($17°$) across the spin range and compare the extracted shadow diameter to the EHT ring measurement.

In [ ]:
spin_values = np.arange(0.0, 0.999, 0.05)
m87_results = []
for a_val in spin_values:
    print(f"\r  M87: a = {a_val:.3f}", end='', flush=True)
    params = {
        'spin': a_val, 'inclination': M87['incl'],
        'width': 512, 'height': 512, 'fov': 7.0,
        'obs_dist': 500, 'step_size': 0.15,
        'bg_mode': 1, 'show_disk': False,
    }
    raw_bytes = renderer.render_frame(params)
    img_gray = np.frombuffer(raw_bytes, dtype=np.uint8).reshape((512, 512, 3))
    obs = nt.extract_shadow_metrics(img_gray, fov_deg=7.0)
    if 'error' not in obs:
        obs['spin'] = a_val
        obs['diameter_uas'] = M_to_uas(obs['diameter_M'], M87['mass_kg'], M87['dist_m'])
        m87_results.append(obs)
print(f"\n  Done — {len(m87_results)} renders.")


In [ ]:
spins_arr = np.array([r['spin'] for r in m87_results])
diam_uas = np.array([r['diameter_uas'] for r in m87_results])
delta_Cs = np.array([r['circularity'] for r in m87_results])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 10), sharex=True,
                               gridspec_kw={'height_ratios': [1.2, 1]})

ax1.plot(spins_arr, diam_uas, 'o-', color='#FF6B35', lw=2.5, ms=5,
         label='Nulltracer (this work)', zorder=5)
ax1.axhspan(M87['ring_uas'] - M87['ring_err'], M87['ring_uas'] + M87['ring_err'],
            alpha=0.25, color='cyan', label=rf"EHT M87* ($42 \pm 3$ μas)")
ax1.axhline(M87['ring_uas'], color='cyan', ls='--', lw=1)
ax1.set_ylabel('Shadow Diameter (μas)')
ax1.legend(fontsize=11); ax1.grid(alpha=0.15)
ax1.set_title(f'M87* Shadow Observables vs. Spin ($\\theta = {M87["incl"]:.0f}°$, '
              'Fuerst & Wu Hamiltonian, RK4)', fontsize=13)

ax2.plot(spins_arr, delta_Cs, 's-', color='#7BD389', lw=2.5, ms=5,
         label='Nulltracer', zorder=5)
ax2.axhline(0.10, color='gold', ls='--', lw=1.5,
            label='EHT bound ($\\Delta C \\lesssim 0.10$)')
ax2.fill_between([0, 1], 0, 0.10, alpha=0.12, color='gold')
ax2.set_xlabel('Spin Parameter $a/M$'); ax2.set_ylabel('$\\Delta C$')
ax2.legend(fontsize=11); ax2.grid(alpha=0.15); ax2.set_xlim(-0.02, 1)

plt.tight_layout()
plt.savefig('m87_spin_sweep.png', bbox_inches='tight')
plt.show()

d_ok = (diam_uas >= M87['ring_uas'] - M87['ring_err']) & \
       (diam_uas <= M87['ring_uas'] + M87['ring_err'])
c_ok = delta_Cs <= 0.10
both = d_ok & c_ok
if both.any():
    print(f"Spin consistent with M87* (diameter + ΔC): "
          f"a ∈ [{spins_arr[both].min():.2f}, {spins_arr[both].max():.2f}]")
else:
    print(f"No spin matches both constraints. Diameter range: "
          f"[{diam_uas.min():.1f}, {diam_uas.max():.1f}] μas")


---

## 8. Comparison to Sgr A\*

| Observable | Value | Reference |
|---|---|---|
| Mass | $4.0^{+1.1}_{-0.6} \times 10^6\;M_\odot$ | EHT Sgr A\* I (2022) |
| Distance | $8.28$ kpc | GRAVITY Collaboration |
| Shadow diameter | $48.7 \pm 7\;\mu\text{as}$ | EHT Sgr A\* I (2022) |
| Ring diameter | $51.8 \pm 2.3\;\mu\text{as}$ | EHT Sgr A\* I (2022) |
| Inclination | $\sim 50°$ | Model-dependent |


In [ ]:
sgra_results = []
for a_val in spin_values:
    print(f"\r  Sgr A*: a = {a_val:.3f}", end='', flush=True)
    params = {
        'spin': a_val, 'inclination': SgrA['incl'],
        'width': 512, 'height': 512, 'fov': 7.0,
        'obs_dist': 500, 'step_size': 0.15,
        'bg_mode': 1, 'show_disk': False,
    }
    raw_bytes = renderer.render_frame(params)
    img_gray = np.frombuffer(raw_bytes, dtype=np.uint8).reshape((512, 512, 3))
    obs = nt.extract_shadow_metrics(img_gray, fov_deg=7.0)
    if 'error' not in obs:
        obs['spin'] = a_val
        obs['diameter_uas'] = M_to_uas(obs['diameter_M'], SgrA['mass_kg'], SgrA['dist_m'])
        sgra_results.append(obs)
print(f"\n  Done — {len(sgra_results)} renders.")

sgra_s = np.array([r['spin'] for r in sgra_results])
sgra_d = np.array([r['diameter_uas'] for r in sgra_results])

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(sgra_s, sgra_d, 'o-', color='#FF6B35', lw=2.5, ms=5,
        label='Nulltracer', zorder=5)
ax.axhspan(SgrA['shadow_uas'] - SgrA['shadow_err'],
           SgrA['shadow_uas'] + SgrA['shadow_err'],
           alpha=0.2, color='cyan',
           label=rf"EHT shadow ($48.7 \pm 7$ μas)")
ax.axhspan(SgrA['ring_uas'] - SgrA['ring_err'],
           SgrA['ring_uas'] + SgrA['ring_err'],
           alpha=0.2, color='#FF69B4',
           label=rf"EHT ring ($51.8 \pm 2.3$ μas)")
ax.set_xlabel('Spin $a/M$'); ax.set_ylabel('Shadow Diameter (μas)')
ax.set_title(f'Sgr A* Shadow vs. Spin ($\\theta = {SgrA["incl"]:.0f}°$)', fontsize=13)
ax.legend(); ax.grid(alpha=0.15); ax.set_xlim(-0.02, 1)
plt.tight_layout()
plt.savefig('sgra_spin_sweep.png', bbox_inches='tight')
plt.show()


---

## 9. Advanced integration methods

### 9.1 Kahan–Li symplectic integrator

For long-duration integrations near the event horizon, Nulltracer provides an 8th-order Kahan–Li symplectic integrator with Sundman time transformation. The Sundman transform relates the affine parameter $\tau$ to proper distance $s$ via

$$d\tau = \frac{\Sigma}{r^2}\,ds,$$

naturally concentrating steps near the horizon where coordinates become singular.

The Kahan–Li composition uses 8 stages with weights $(w_i, d_i)$ from Kahan & Li (1997):

| $i$ | $w_i$ | $d_i$ |
|---|---|---|
| 0 | 0.741670364350613 | 0.370835182175306 |
| 1 | -0.409100825800032 | 0.166284769275291 |
| 2 | 0.190754710296238 | -0.109173057751897 |
| 3 | -0.573862471116082 | -0.191553880409922 |
| 4 | 0.299064181303656 | -0.137399144906213 |
| 5 | 0.334624918245298 | 0.316844549774477 |
| 6 | 0.315293092396767 | 0.324959005321032 |
| 7 | -0.796887939352916 | -0.240797423478075 |

**Why it matters here:** symplecticity preserves the phase-space volume over many orbital periods, which matters for the sub-leading photon-ring images ($n \geq 1$) that the thin-ring EHT programme is targeting.

### 9.2 Kerr-Newman extension (charged black holes)

The Kerr-Newman line element extends Kerr by adding electric charge $Q$:

$$ds^2 = -\left(1 - \frac{2r - Q^2}{\Sigma}\right)dt^2 + \frac{2a(2r - Q^2)\sin^2\theta}{\Sigma}\,dt\,d\phi + \frac{\Sigma}{\Delta}\,dr^2 + \Sigma\,d\theta^2 + \sin^2\theta\left(r^2 + a^2 + \frac{a^2(2r - Q^2)\sin^2\theta}{\Sigma}\right)d\phi^2,$$

with $\Delta = r^2 - 2r + a^2 + Q^2$. The event horizon moves inward:

$$r_+ = 1 + \sqrt{1 - a^2 - Q^2}.$$

For $Q > 0$ the shadow shrinks relative to Kerr — a potential (if astrophysically negligible) probe of extra-metric charge.

### 9.3 Volumetric emission & jets

Beyond the thin-disk model, Nulltracer's kernel supports volumetric emitters:

1. **Corona** — hot electron scattering layer above the disk with scale height $h = 0.3\,r_{\rm cyl}$.
2. **Relativistic jet** — collimated outflow along the spin axis, intensity $\propto \exp\!\bigl[-(1 - \cos\theta)^2 / 0.003\bigr]$.

### 9.4 Post-processing: Airy-disk bloom

`nulltracer.bloom.apply_bloom` convolves the linear-light image with the diffraction pattern of a circular aperture,

$$I(r) = \left[\frac{2 J_1(kr)}{kr}\right]^2,$$

where $J_1$ is the Bessel function of the first kind. This adds a physically motivated halo around bright features — the ring rather than generic Gaussian bloom.

**See also:** `nulltracer_portfolio.ipynb` demonstrates interactive parameter exploration using the advanced features above.

---

## 10. Kerr characteristic radii

Event horizon, photon sphere (prograde/retrograde), and ISCO as functions of spin — the canonical (Bardeen, Press & Teukolsky 1972) plot.

In [ ]:
a_dense = np.linspace(0, 0.998, 300)
r_isco_arr = np.array([nt.isco_kerr(a) for a in a_dense])
r_ph_pro = 2 * (1 + np.cos(2 / 3 * np.arccos(-a_dense)))
r_ph_ret = 2 * (1 + np.cos(2 / 3 * np.arccos(+a_dense)))
r_hor = 1 + np.sqrt(1 - a_dense ** 2)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(a_dense, r_isco_arr, '-', color='#FF6B35', lw=2.5, label='ISCO (prograde)')
ax.plot(a_dense, r_ph_pro, '-', color='cyan', lw=2, label='Photon orbit (prograde)')
ax.plot(a_dense, r_ph_ret, '--', color='cyan', lw=1.5, alpha=0.5,
        label='Photon orbit (retrograde)')
ax.plot(a_dense, r_hor, '--', color='#888', lw=1.5, label='Event horizon $r_+$')
ax.set_xlabel('Spin $a/M$'); ax.set_ylabel('Radius ($M$)')
ax.set_title('Kerr Characteristic Radii (Bardeen, Press & Teukolsky 1972)', fontsize=13)
ax.legend(); ax.grid(alpha=0.15); ax.set_xlim(0, 1); ax.set_ylim(0, 10)
plt.tight_layout()
plt.savefig('kerr_radii.png', bbox_inches='tight')
plt.show()


---

## 11. Integrator comparison and observer in-fall

### 11.1 Side-by-side integrator comparison

In [ ]:
results, fig = nt.compare_integrators(
    spin=0.9, inclination_deg=70, obs_dist=40, step_size=0.3,
    width=512, height=512, disk_temp=1.3, bg_mode=3,
)
plt.show()


### 11.2 Falling into a Kerr black hole

Observer distance $r$ decreases toward $r_+$. FOV widens to compensate; below $r_+$ the rendering has no physical meaning.

In [ ]:
distances = [200, 100, 50, 30, 20, 15, 10, 7, 5, 4, 3.5, 3.0]
spin_fall, incl_fall = 0.9, 75
rp_fall = 1 + np.sqrt(1 - spin_fall ** 2)

fig, axes = plt.subplots(3, 4, figsize=(18, 13.5))
for ax, d in zip(axes.flat, distances):
    params = {
        'spin': spin_fall, 'inclination': incl_fall,
        'width': 384, 'height': 384,
        'fov': min(7.0 + 30.0 / d, 40.0),
        'bg_mode': 3, 'obs_dist': d,
        'step_size': 0.15, 'method': 'rk4',
    }
    timed = renderer.render_frame_timed(params)
    img = np.frombuffer(timed['raw_rgb'], dtype=np.uint8).reshape((384, 384, 3))
    kernel_ms = timed["kernel_ms"]        # extract first — avoid nested quote issues
    ax.imshow(img)
    ax.set_title(
        f"$r = {d}\\,M$ ({d / rp_fall:.1f} $r_+$)\n{kernel_ms:.0f} ms",
        fontsize=10,
    )
    ax.axis('off')

fig.suptitle(f'Falling into a Kerr black hole ($a={spin_fall}$, '
             rf'$\theta={incl_fall}°$)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


---

## 12. Discussion & Conclusions

### Summary

- Implemented the Fuerst & Wu (2004) Hamiltonian formulation of Kerr null geodesics as a CuPy CUDA `RawKernel`, matching the approach used by production EHT codes (Odyssey, GRay, RAPTOR).
- Validated the numerical shadow boundary against the exact Bardeen (1973) analytic curve for four test cases spanning $a \in [0, 0.998]$ and $\theta \in [17°, 90°]$.
- Verified the Schwarzschild shadow diameter recovers $2 \times 3\sqrt{3} \approx 10.39\,M$ to within numerical precision.
- Parameter sweeps over spin at M87\*'s inclination ($17°$) and Sgr A\*'s inclination ($50°$) produce shadow sizes consistent with published EHT measurements within their stated uncertainties.
- Disk emission includes the correct relativistic redshift factor $g$ for $r \geq r_{\rm ISCO}$ with a plunging-region continuation inside ISCO, and bolometric $g^4$ boosting for optically thick emission.

### Limitations

1. **Geometric thin-disk model.** The EHT papers use GRMHD simulations with turbulent magnetized plasma. This work uses an analytic Novikov–Thorne-like temperature profile in the equatorial plane.
2. **Image-domain comparison only.** A full EHT comparison computes synthetic complex visibilities and fits in the $(u,v)$ Fourier plane.
3. **No interstellar scattering** (relevant for Sgr A\*) and no time variability.
4. **Single-crossing default.** By default only the first equatorial crossing is recorded per ray; `disk_max_crossings > 1` is required to build up the photon-ring sub-images ($n = 1, 2, \ldots$).
5. **Docstring unit mismatch.** The `fov` parameter is documented as "degrees" in `render_frame` but the kernel treats it as screen half-width in units of $M$. The two are consistent in this notebook (the conversion in §4 uses the M-sense, matching the kernel).

### Future Directions

- Accumulate multiple disk crossings for photon-ring sub-images ($n = 1, 2, \ldots$) and compare against the universal $e^{-\pi}$ demagnification.
- Compute synthetic visibility amplitudes for direct comparison to EHT $(u,v)$-plane data.
- MCMC parameter estimation over $(a, \theta, M)$ against the published EHT ring-diameter posteriors.
- Extend `shadow_boundary` to the full Kerr–Newman case so the charged-BH narrative in §9.2 can be validated as tightly as Kerr is here.

---

**Nulltracer source code:** [github.com/ethank5149/nulltracer](https://github.com/ethank5149/nulltracer)  
**Contact:** ethank5149@gmail.com

### References

- Bardeen, J. M. (1973). *Timelike and null geodesics in the Kerr metric.* In *Black Holes*, DeWitt & DeWitt (eds).
- Bardeen, J. M., Press, W. H., & Teukolsky, S. A. (1972). *Rotating black holes.* ApJ **178**, 347.
- Bronzwaer, T. et al. (2018). *RAPTOR I: Time-dependent radiative transfer.* A&A **613**, A2.
- Carter, B. (1968). *Global structure of the Kerr family.* Phys. Rev. **174**, 1559.
- Chan, C. et al. (2013). *GRay: A massively parallel GPU-based code for ray tracing in Kerr spacetime.* ApJ **777**, 13.
- Chandrasekhar, S. (1983). *The Mathematical Theory of Black Holes.* Oxford Univ. Press.
- EHT Collaboration (2019). *First M87 EHT Results. I–VI.* ApJL **875**.
- EHT Collaboration (2022). *First Sgr A\* EHT Results. I–VI.* ApJL **930**.
- Fuerst, S. V. & Wu, K. (2004). *Radiation transfer of emission lines in curved space-time.* A&A **424**, 733.
- Kahan, W. & Li, R.-C. (1997). *Composition constants for raising the orders of unconventional schemes for ordinary differential equations.* Math. Comp. **66**, 1089.
- Page, D. N. & Thorne, K. S. (1974). *Disk-accretion onto a black hole.* ApJ **191**, 499.
- Pu, H.-Y. et al. (2016). *Odyssey: A public GPU-based code for general-relativistic radiative transfer in Kerr spacetime.* ApJ **820**, 105.
